# Week 4 Exercise — Python → Go Code Converter

Convert Python code into idiomatic, high-performance **Go (Golang)** using frontier LLMs.

### Models
| Model | Provider | Best for |
|---|---|---|
| Claude Haiku 4.5 | Anthropic | Fast, cost-effective conversions |
| Claude Sonnet 4.5 | Anthropic | Higher quality, complex code |
| DeepSeek Chat | DeepSeek | Strong code generation, low cost |

### Features
- Side-by-side Python and Go code editors
- Run Python directly to see its output
- Convert Python to Go using your chosen LLM
- Copy the generated Go code to run elsewhere

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

# ── API clients ───────────────────────────────────────────────────────────────
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key  = os.getenv('DEEPSEEK_API_KEY')

anthropic = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
)
deepseek = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com/v1"
)

if anthropic_api_key:
    print(f"Anthropic API key found: {anthropic_api_key[:7]}...")
else:
    print("Anthropic API key NOT set")

if deepseek_api_key:
    print(f"DeepSeek API key found:  {deepseek_api_key[:6]}...")
else:
    print("DeepSeek API key NOT set")

# ── Model registry ────────────────────────────────────────────────────────────
MODELS = {
    "DeepSeek Chat": {
        "client": deepseek,
        "model":  "deepseek-chat"
    },
    "Claude Haiku 4.5 (Anthropic)": {
        "client": anthropic,
        "model":  "claude-haiku-4-5"
    },
    "Claude Sonnet 4.5 (Anthropic)": {
        "client": anthropic,
        "model":  "claude-sonnet-4-5-20250929"
    },
}

# ── Prompts ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are an expert Go (Golang) developer.
Your task is to convert Python code into idiomatic, high-performance Go code.

Rules:
- Respond ONLY with Go source code — no markdown fences, no explanations.
- Always include `package main` and all required imports.
- The program must produce output identical to the Python original.
- Use idiomatic Go: proper error handling, standard library, efficient data structures.
- The code must compile and run with: go run main.go
"""

def user_prompt_for(python: str) -> str:
    return (
        "Convert the following Python code to idiomatic Go.\n"
        "The Go program must produce identical output.\n"
        "Include package main and all imports.\n"
        "Respond ONLY with the Go source code.\n\n"
        f"Python code:\n\n{python}"
    )

def messages_for(python: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt_for(python)}
    ]


# ── Conversion function ───────────────────────────────────────────────────────
def convert_to_go(model_name: str, python: str) -> str:
    """Call the selected LLM and return clean Go source code."""
    if not python.strip():
        return "# Paste some Python code on the left to convert."
    cfg = MODELS[model_name]
    try:
        response = cfg["client"].chat.completions.create(
            model=cfg["model"],
            messages=messages_for(python),
            temperature=0.2,
        )
        code = response.choices[0].message.content
        # Strip any markdown fences the model may have added
        for fence in ("```go", "```golang", "```"):
            code = code.replace(fence, "")
        return code.strip()
    except Exception as e:
        return f"// Error during conversion: {e}"


# ── Run Python (in-process) ───────────────────────────────────────────────────
def run_python(python: str) -> str:
    """Execute Python code and capture stdout."""
    if not python.strip():
        return ""
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(python, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Python Error:\n{e}"
    finally:
        sys.stdout = old_stdout




# ── CSS ───────────────────────────────────────────────────────────────────────
# Python blue: #3776AB   |   Go cyan: #00ADD8
CSS = """
:root {
  --py-color: #3776AB;
  --go-color: #00ADD8;
  --accent:   #00ADD8;
  --card:     #161a22;
  --text:     #e9eef5;
}

.gradio-container {
  max-width: 100% !important;
  padding: 0 40px !important;
}

.convert-btn button {
  background: var(--accent) !important;
  border-color: rgba(255,255,255,.12) !important;
  color: #111 !important;
  font-weight: 700;
}
.run-btn button {
  background: #202631 !important;
  color: var(--text) !important;
  border-color: rgba(255,255,255,.12) !important;
}
.run-btn.py button:hover  { box-shadow: 0 0 0 2px var(--py-color) inset; }
.run-btn.go button:hover  { box-shadow: 0 0 0 2px var(--go-color) inset; }
.convert-btn button:hover { box-shadow: 0 0 0 2px var(--card) inset; }

.py-out textarea {
  background: linear-gradient(180deg, rgba(55,118,171,.18), rgba(55,118,171,.10));
  border: 1px solid rgba(55,118,171,.45) !important;
  color: rgba(100,180,255,1) !important;
  font-weight: 600;
}
.go-out textarea {
  background: linear-gradient(180deg, rgba(0,173,216,.22), rgba(0,173,216,.12));
  border: 1px solid rgba(0,173,216,.45) !important;
  color: rgba(0,220,255,1) !important;
  font-weight: 600;
}

.controls .wrap {
  gap: 10px;
  justify-content: center;
  align-items: center;
}
"""

# ── Sample Python code ────────────────────────────────────────────────────────
SAMPLE_PYTHON = '''\
import time
from collections import Counter

def word_frequency(text):
    words = text.lower().split()
    return Counter(words)

def fibonacci(n):
    a, b = 0, 1
    result = []
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

start = time.time()

# Word frequency
text = "the quick brown fox jumps over the lazy dog the fox"
freq = word_frequency(text)
print("Word frequencies (top 5):")
for word, count in sorted(freq.items(), key=lambda x: -x[1])[:5]:
    print(f"  {word}: {count}")

# Fibonacci sequence
fibs = fibonacci(15)
print('')
print('First 15 Fibonacci numbers:')
print(fibs)

# Prime numbers
primes = [n for n in range(2, 51) if is_prime(n)]
print('')
print('Primes up to 50:')
print(primes)

elapsed = time.time() - start
print('')
print('Done in ' + str(round(elapsed, 6)) + 's')
'''

# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Python → Go Converter") as ui:

    gr.Markdown(
        "# Python → Go Converter\n"
        "Translate Python into idiomatic Go using **Anthropic Claude** or **DeepSeek**. "
        "Run both and compare outputs to verify correctness."
    )

    # ── Code panels ───────────────────────────────────────────────────────────
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_code = gr.Code(
                label="Python (original)",
                value=SAMPLE_PYTHON,
                language="python",
                lines=28
            )
        with gr.Column(scale=6):
            go_code = gr.Code(
                label="Go (generated)",
                value="",
                language=None,
                lines=28
            )

    # ── Control bar ───────────────────────────────────────────────────────────
    with gr.Row(elem_classes=["controls"]):
        python_run_btn = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="DeepSeek Chat",
            show_label=False
        )
        convert_btn = gr.Button("Convert to Go  →", elem_classes=["convert-btn"])

    # ── Output panel (Python only) ────────────────────────────────────────────
    python_out = gr.TextArea(
        label="Python output",
        lines=10,
        elem_classes=["py-out"]
    )

    # ── Wire up events ────────────────────────────────────────────────────────
    convert_btn.click(
        fn=convert_to_go,
        inputs=[model_dd, python_code],
        outputs=[go_code]
    )
    python_run_btn.click(
        fn=run_python,
        inputs=[python_code],
        outputs=[python_out]
    )

ui.launch(inbrowser=True)
